# 06 Backtesting - Kupiec & Christoffersen Tests

Statistical validation of VaR forecasts using POF and independence tests.

In [ ]:
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.volatility_forecasting.data.loader import download_data
from src.volatility_forecasting.data.preprocessor import DataPreprocessor
from src.volatility_forecasting.models.garch_models import rolling_volatility_forecast
from src.volatility_forecasting.analysis.var_analysis import compute_var_es
from src.volatility_forecasting.analysis.backtesting import kupiec_test, christoffersen_test
from src.volatility_forecasting.config import DATA_START_DATE, DATA_END_DATE, TICKER, VAR_CONFIDENCE_LEVELS
from src.volatility_forecasting.logger import setup_logger

logger = setup_logger('notebook')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Load data
price_data = download_data(ticker=TICKER, start=DATA_START_DATE, end=DATA_END_DATE)
preprocessor = DataPreprocessor(price_data)
returns = preprocessor.compute_log_returns().dropna() * 100

# Split train/test
split_idx = int(len(returns) * 0.8)
train = returns.iloc[:split_idx]
test = returns.iloc[split_idx:]

print(f"Test period: {len(test)} observations")

In [ ]:
# Generate VaR forecasts
vol_forecast = rolling_volatility_forecast(
    full_returns=returns,
    test_index=test.index,
    train_size=len(train),
    vol='Garch', p=1, o=1, q=1, dist='t'
)

var_results = compute_var_es(
    vol_forecast,
    confidence_levels=VAR_CONFIDENCE_LEVELS,
    dist='t', df_t=5.0
)

print("VaR forecasts generated")

In [ ]:
# Kupiec POF (Proportion of Failures) Test
print("="*60)
print("KUPIEC POF TEST (Unconditional Coverage)")
print("="*60)

kupiec_results = []

for conf in VAR_CONFIDENCE_LEVELS:
    var_col = f'VaR_{int(conf*100)}'
    lr_stat, p_value, passed = kupiec_test(
        actual=test.values,
        var_series=var_results[var_col].values,
        confidence=conf
    )
    
    # Count violations
    violations = np.sum(test.values < -var_results[var_col].values)
    violation_rate = violations / len(test) * 100
    expected_rate = (1 - conf) * 100
    
    kupiec_results.append({
        'Confidence': f'{int(conf*100)}%',
        'Expected Violations': f'{expected_rate:.2f}%',
        'Actual Violations': f'{violation_rate:.2f}%',
        'LR Statistic': f'{lr_stat:.4f}',
        'P-value': f'{p_value:.4f}',
        'Pass (p>0.05)': 'Yes' if passed else 'No'
    })
    
    print(f"\n{int(conf*100)}% Confidence Level:")
    print(f"  Expected violations: {expected_rate:.2f}%")
    print(f"  Actual violations: {violations} ({violation_rate:.2f}%)")
    print(f"  LR Statistic: {lr_stat:.4f}")
    print(f"  P-value: {p_value:.4f}")
    print(f"  Result: {'PASS' if passed else 'FAIL'} (p {'>' if passed else '<'} 0.05)")

kupiec_df = pd.DataFrame(kupiec_results)
print("\n" + kupiec_df.to_string(index=False))

In [ ]:
# Christoffersen Independence Test
print("\n" + "="*60)
print("CHRISTOFFERSEN INDEPENDENCE TEST")
print("="*60)

chris_results = []

for conf in VAR_CONFIDENCE_LEVELS:
    var_col = f'VaR_{int(conf*100)}'
    lr_stat, p_value, passed = christoffersen_test(
        actual=test.values,
        var_series=var_results[var_col].values
    )
    
    chris_results.append({
        'Confidence': f'{int(conf*100)}%',
        'LR Statistic': f'{lr_stat:.4f}',
        'P-value': f'{p_value:.4f}',
        'Independent (p>0.05)': 'Yes' if passed else 'No'
    })
    
    print(f"\n{int(conf*100)}% Confidence Level:")
    print(f"  LR Statistic: {lr_stat:.4f}")
    print(f"  P-value: {p_value:.4f}")
    print(f"  Result: {'PASS (violations independent)' if passed else 'FAIL (violations clustered)'}")

chris_df = pd.DataFrame(chris_results)
print("\n" + chris_df.to_string(index=False))

In [ ]:
# Visualize violations
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 95% VaR
violations_95 = test.values < -var_results['VaR_95'].values
axes[0].plot(test.index, test.values, label='Actual Returns', alpha=0.7)
axes[0].axhline(-var_results['VaR_95'].mean(), color='r', linestyle='--', label='Mean VaR 95%')
axes[0].scatter(
    test.index[violations_95],
    test.values[violations_95],
    color='red', s=50, label=f'Violations ({violations_95.sum()})',
    zorder=5
)
axes[0].set_title('95% VaR Violations')
axes[0].set_ylabel('Return (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 99% VaR
violations_99 = test.values < -var_results['VaR_99'].values
axes[1].plot(test.index, test.values, label='Actual Returns', alpha=0.7)
axes[1].axhline(-var_results['VaR_99'].mean(), color='darkred', linestyle='--', label='Mean VaR 99%')
axes[1].scatter(
    test.index[violations_99],
    test.values[violations_99],
    color='darkred', s=50, label=f'Violations ({violations_99.sum()})',
    zorder=5
)
axes[1].set_title('99% VaR Violations')
axes[1].set_ylabel('Return (%)')
axes[1].set_xlabel('Date')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../report/figures/06_backtesting.png', dpi=150, bbox_inches='tight')
plt.show()

logger.info("Backtesting analysis complete")